# 00 — Route Pre-computation

Computes the shortest route between **every directed port pair** in the network and saves the
results permanently. `01_ship_generation.ipynb` loads these files instead of recomputing.

**Prerequisites:** Run `simulation_config.ipynb` first, and ensure the calibrated network
(`network_calibrated.gpickle`) has been produced by `network_calibration.ipynb`.

**Outputs** (written to `cfg['OUTPUT_DIR']`, e.g. `simulation_output_data/scenario_baseline/`):
- `port_pair_routes.pkl` — `{(portname_A, portname_B): {'path': [...], 'length': float}}`
- `country_pair_optimal.pkl` — `{(country_A, country_B): {'origin_port', 'dest_port', 'optimal_length'}}`

**Scenario control:** To change which subfolder outputs go into, edit `OUTPUT_DIR` in
`simulation_config.ipynb` and re-run it to update `simulation_config.json`. All four
notebooks (`00` through `02`) read from the same `OUTPUT_DIR`.

**Runtime:** Scales as O(n²) in the number of ports.  
At 592 ports (75 % coverage threshold) expect ≈ 3 hours.

---

Re-run this notebook only if the network changes (new ports, re-calibration).
The output files are safe to reuse across different simulation runs.

In [7]:
import pickle
import sys
from pathlib import Path

import networkx as nx

# Add simulation_engine to path
sys.path.insert(0, str(Path('.').resolve()))

from simulation_engine.config_loader import load_config, resolve_paths
from simulation_engine.routing import (
    build_port_node_map,
    build_country_port_map,
    compute_all_port_pair_routes,
    derive_country_pair_optimal,
)

print('Imports loaded successfully.')

Imports loaded successfully.


In [8]:
# ============================================================
# Load configuration
# ============================================================
cfg = load_config()
cfg = resolve_paths(cfg)

output_dir = Path(cfg['OUTPUT_DIR'])
output_dir.mkdir(parents=True, exist_ok=True)

PORT_PAIR_ROUTES_PATH    = output_dir / 'port_pair_routes.pkl'
COUNTRY_OPTIMAL_PATH     = output_dir / 'country_pair_optimal.pkl'

print('Configuration loaded.')
print(f'Output directory: {output_dir}')
print(f'port_pair_routes.pkl  → {PORT_PAIR_ROUTES_PATH}')
print(f'country_pair_optimal.pkl → {COUNTRY_OPTIMAL_PATH}')

Configuration loaded.
Output directory: /Users/finn-frederikhopmann/Documents/Minerva/Academics/Year 4/Semester 7/CP193/Data Fun/simulation_pipeline/part_4_new_simulation/simulation_output_data
port_pair_routes.pkl  → /Users/finn-frederikhopmann/Documents/Minerva/Academics/Year 4/Semester 7/CP193/Data Fun/simulation_pipeline/part_4_new_simulation/simulation_output_data/port_pair_routes.pkl
country_pair_optimal.pkl → /Users/finn-frederikhopmann/Documents/Minerva/Academics/Year 4/Semester 7/CP193/Data Fun/simulation_pipeline/part_4_new_simulation/simulation_output_data/country_pair_optimal.pkl


In [9]:
# ============================================================
# Load network
# ============================================================
print('Loading network...')
with open(cfg['NETWORK_FILE'], 'rb') as f:
    G = pickle.load(f)

port_name_to_node = build_port_node_map(G)
country_to_ports  = build_country_port_map(G)

n_ports    = len(port_name_to_node)
n_countries = len(country_to_ports)
n_pairs    = n_ports * (n_ports - 1)

print(f'  Nodes: {G.number_of_nodes()}  |  Edges: {G.number_of_edges()}')
print(f'  Ports: {n_ports}  |  Countries: {n_countries}')
print(f'  Directed port pairs to compute: {n_pairs:,}')

Loading network...
  Nodes: 8726  |  Edges: 14634
  Ports: 588  |  Countries: 168
  Directed port pairs to compute: 345,156


In [10]:
# ============================================================
# Compute all port-pair routes
# ============================================================
# This cell runs the A* pathfinding for every directed port pair.
# Estimated runtime: ~3 h for 592 ports (varies by machine and graph density).
#
# If the output file already exists, this cell loads it from disk instead.
# Delete port_pair_routes.pkl to force a recompute.

if PORT_PAIR_ROUTES_PATH.exists():
    print(f'Loading existing port_pair_routes from: {PORT_PAIR_ROUTES_PATH}')
    with open(PORT_PAIR_ROUTES_PATH, 'rb') as f:
        port_pair_routes = pickle.load(f)
    print(f'Loaded {len(port_pair_routes):,} routes.')
else:
    print('Computing port-pair routes (this will take a while)...')
    port_pair_routes = compute_all_port_pair_routes(G, port_name_to_node, show_progress=True)

    print(f'\nComputed {len(port_pair_routes):,} routes out of {n_pairs:,} pairs.')
    print(f'Unreachable pairs: {n_pairs - len(port_pair_routes):,}')

    with open(PORT_PAIR_ROUTES_PATH, 'wb') as f:
        pickle.dump(port_pair_routes, f)
    print(f'Saved to: {PORT_PAIR_ROUTES_PATH}')
    print(f'File size: {PORT_PAIR_ROUTES_PATH.stat().st_size / 1024 / 1024:.1f} MB')

Computing port-pair routes (this will take a while)...


Computing port-pair routes: 100%|██████████| 345156/345156 [47:43<00:00, 120.54pair/s] 



Computed 345,156 routes out of 345,156 pairs.
Unreachable pairs: 0
Saved to: /Users/finn-frederikhopmann/Documents/Minerva/Academics/Year 4/Semester 7/CP193/Data Fun/simulation_pipeline/part_4_new_simulation/simulation_output_data/port_pair_routes.pkl
File size: 111.5 MB


In [11]:
# ============================================================
# Derive country-pair optimal routes
# ============================================================
# For each (origin_country, dest_country) pair, find the port pair
# with the shortest route. This is the reference distance used in
# the distance-penalty formula during ship generation.

print('Deriving country-pair optimal routes...')
country_pair_optimal = derive_country_pair_optimal(port_pair_routes, country_to_ports)

n_country_pairs = len(country_to_ports) * (len(country_to_ports) - 1)
print(f'Country pairs with at least one route: {len(country_pair_optimal):,} / {n_country_pairs:,}')

with open(COUNTRY_OPTIMAL_PATH, 'wb') as f:
    pickle.dump(country_pair_optimal, f)
print(f'Saved to: {COUNTRY_OPTIMAL_PATH}')

Deriving country-pair optimal routes...
Country pairs with at least one route: 28,056 / 28,056
Saved to: /Users/finn-frederikhopmann/Documents/Minerva/Academics/Year 4/Semester 7/CP193/Data Fun/simulation_pipeline/part_4_new_simulation/simulation_output_data/country_pair_optimal.pkl


In [12]:
# ============================================================
# Summary
# ============================================================
import numpy as np

lengths = [v['length'] for v in port_pair_routes.values()]

print('=' * 50)
print('ROUTE PRE-COMPUTATION SUMMARY')
print('=' * 50)
print(f'Ports:              {n_ports}')
print(f'Countries:          {n_countries}')
print(f'Port-pair routes:   {len(port_pair_routes):,}')
print(f'Country-pair opt.:  {len(country_pair_optimal):,}')
print()
print('Route length distribution (km):')
print(f'  Min:    {np.min(lengths):,.0f}')
print(f'  Median: {np.median(lengths):,.0f}')
print(f'  Mean:   {np.mean(lengths):,.0f}')
print(f'  Max:    {np.max(lengths):,.0f}')
print()
print('Output files:')
print(f'  {PORT_PAIR_ROUTES_PATH}')
print(f'  {COUNTRY_OPTIMAL_PATH}')
print()
print('Next step: run 01_ship_generation.ipynb')

ROUTE PRE-COMPUTATION SUMMARY
Ports:              588
Countries:          168
Port-pair routes:   345,156
Country-pair opt.:  28,056

Route length distribution (km):
  Min:    5
  Median: 11,501
  Mean:   11,905
  Max:    26,363

Output files:
  /Users/finn-frederikhopmann/Documents/Minerva/Academics/Year 4/Semester 7/CP193/Data Fun/simulation_pipeline/part_4_new_simulation/simulation_output_data/port_pair_routes.pkl
  /Users/finn-frederikhopmann/Documents/Minerva/Academics/Year 4/Semester 7/CP193/Data Fun/simulation_pipeline/part_4_new_simulation/simulation_output_data/country_pair_optimal.pkl

Next step: run 01_ship_generation.ipynb
